# Notebook 15 - RQ3 deep ensemble comparison

RQ3 evidence, first half. The proposal (Section 2.3, 3.3) tests evidential uncertainty against a
deep ensemble on calibration, in distribution and under leave-one-code-out shift. Five
independently seeded MT-TrajNet instances are trained on fold 1, as the proposal specifies.

Both methods are calibrated identically, using the calibration/evaluation partition split the
supervisor specified: the fold-1 test set is split in half, the uncertainty scale is fitted on
the calibration half so nominal 90 percent coverage is reached, and PICP, ECE and the
error-uncertainty correlation are reported on the evaluation half only. The ensemble predictive
variance is the between-seed spread of the means plus the mean of the per-seed aleatoric
variances, which is the standard ensemble predictive distribution.

Self-contained, MD5-verified, writes only rq3_ensemble_comparison.json. Evidential baselines are
recomputed from the authoritative oof_predictions.npz using the same partition method.

In [2]:
import numpy as np,pickle,torch,hashlib,time,json
import torch.nn as nn,torch.nn.functional as F
from scipy.stats import norm
from scipy.optimize import brentq

SEED=42
np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
device="cuda" if torch.cuda.is_available() else "cpu"
print("device:",device,"|",torch.cuda.get_device_name(0) if device=="cuda" else "cpu")

DATA="/kaggle/input/datasets/arpitjoshua/mt-trajnet-thesis-data/kaggle_upload"
PKL=DATA+"/assembled_trajectories.pkl"
h=hashlib.md5()
with open(PKL,"rb") as fh:
    for c in iter(lambda:fh.read(1<<20),b""): h.update(c)
md5=h.hexdigest()
print("pkl md5:",md5)
assert md5=="7d7fc76be1e4940198b76d9d0797a3a9","PKL HASH MISMATCH - STOP"
print("pkl verified")

with open(PKL,"rb") as fh: d=pickle.load(fh)
X_traj=d["X_traj"];y_arr=d["y_arr"];groups=d["groups"]
assert len(X_traj)==1005

STRIDE=2;MAXLEN=6000
targets=["dissolution_av","tbl_av_hardness","tbl_rsd_weight","fct_tensile"]
def prep_batch(traj_list,mean,std,maxlen=MAXLEN):
    out=[]
    for a in traj_list:
        a=a[::STRIDE];a=(a-mean)/std
        if len(a)<maxlen:
            pad=np.zeros((maxlen-len(a),a.shape[1]),dtype="float32");a=np.vstack([a,pad])
        else: a=a[:maxlen]
        out.append(a)
    return torch.tensor(np.array(out,dtype="float32")).transpose(1,2)

class TCNBlock(nn.Module):
    def __init__(self,in_ch,out_ch,dilation,kernel=3,dropout=0.1):
        super().__init__()
        pad=(kernel-1)*dilation
        self.conv1=nn.Conv1d(in_ch,out_ch,kernel,padding=pad,dilation=dilation)
        self.conv2=nn.Conv1d(out_ch,out_ch,kernel,padding=pad,dilation=dilation)
        self.relu=nn.ReLU();self.drop=nn.Dropout(dropout);self.pad=pad
        self.down=nn.Conv1d(in_ch,out_ch,1) if in_ch!=out_ch else None
    def forward(self,x):
        res=x if self.down is None else self.down(x)
        out=self.conv1(x)[:,:,:-self.pad];out=self.drop(self.relu(out))
        out=self.conv2(out)[:,:,:-self.pad];out=self.drop(self.relu(out))
        return self.relu(out+res)

class TCNEncoder(nn.Module):
    def __init__(self,in_ch=10,hidden=128,dropout=0.1):
        super().__init__()
        self.b1=TCNBlock(in_ch,hidden,1,dropout=dropout);self.b2=TCNBlock(hidden,hidden,2,dropout=dropout)
        self.b3=TCNBlock(hidden,hidden,4,dropout=dropout);self.b4=TCNBlock(hidden,hidden,8,dropout=dropout)
    def forward(self,x):
        x=self.b1(x);x=self.b2(x);x=self.b3(x);x=self.b4(x);return x.mean(dim=2)

class EvidentialHead(nn.Module):
    def __init__(self,in_dim,hidden=64,dropout=0.1):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(in_dim,hidden),nn.ReLU(),nn.Dropout(dropout),nn.Linear(hidden,4))
    def forward(self,x):
        out=self.net(x)
        gamma=out[:,0]
        nu=F.softplus(out[:,1]).clamp(min=1e-2,max=1e3)
        alpha=F.softplus(out[:,2]).clamp(min=1e-2,max=1e3)+1.0
        beta=F.softplus(out[:,3]).clamp(min=1e-2,max=1e3)
        return gamma,nu,alpha,beta

def evidential_loss(y,gamma,nu,alpha,beta,lam=1.0):
    om=2*beta*(1+nu)
    nll=(0.5*torch.log(np.pi/nu)-alpha*torch.log(om)+(alpha+0.5)*torch.log(nu*(y-gamma)**2+om)
         +torch.lgamma(alpha)-torch.lgamma(alpha+0.5))
    reg=torch.abs(y-gamma)*(2*nu+alpha)
    return (nll+lam*reg).mean()

class MTTrajNet(nn.Module):
    def __init__(self,in_ch=10,hidden=128,n_targets=4,dropout=0.1):
        super().__init__()
        self.encoder=TCNEncoder(in_ch,hidden,dropout)
        self.heads=nn.ModuleList([EvidentialHead(hidden,dropout=dropout) for _ in range(n_targets)])
    def forward(self,x):
        z=self.encoder(x);outs=[h(z) for h in self.heads]
        gamma=torch.stack([o[0] for o in outs],dim=1);nu=torch.stack([o[1] for o in outs],dim=1)
        alpha=torch.stack([o[2] for o in outs],dim=1);beta=torch.stack([o[3] for o in outs],dim=1)
        return gamma,nu,alpha,beta

fold_codes=[[1,2,4,6,7,9,13,18,25],[10,11,17,19,22,24],[3,5,8,12,14,16,20,21,23]]
OOD_CODE=15;n_splits=3
fold_test_idx=[np.where(np.isin(groups,fold_codes[f]))[0] for f in range(n_splits)]
te=fold_test_idx[0]
trv=np.concatenate([fold_test_idx[j] for j in range(n_splits) if j!=0])
gg=groups[trv];uniq=np.unique(gg)
rng=np.random.RandomState(SEED+0)
val_codes=rng.choice(uniq,size=max(1,len(uniq)//8),replace=False)
va=trv[np.isin(gg,val_codes)];tr=trv[~np.isin(gg,val_codes)]
ood_idx=np.where(groups==OOD_CODE)[0]
assert len(te)==314 and len(ood_idx)==64
print("fold-1 train",len(tr),"val",len(va),"test",len(te),"ood",len(ood_idx))

def train_one(seed,max_epochs=150,lr=5e-4,batch_size=16,patience=10):
    xmean=np.vstack([X_traj[i][::STRIDE] for i in tr]).mean(axis=0)
    xstd=np.vstack([X_traj[i][::STRIDE] for i in tr]).std(axis=0)+1e-6
    Xtr=prep_batch([X_traj[i] for i in tr],xmean,xstd)
    Xva=prep_batch([X_traj[i] for i in va],xmean,xstd)
    Xte=prep_batch([X_traj[i] for i in te],xmean,xstd)
    Xood=prep_batch([X_traj[i] for i in ood_idx],xmean,xstd)
    ymean=y_arr[tr].mean(axis=0);ystd=y_arr[tr].std(axis=0)+1e-6
    ytr=torch.tensor((y_arr[tr]-ymean)/ystd,dtype=torch.float32)
    yva=torch.tensor((y_arr[va]-ymean)/ystd,dtype=torch.float32)
    torch.manual_seed(seed)
    model=MTTrajNet().to(device);opt=torch.optim.Adam(model.parameters(),lr=lr)
    best=float("inf");best_state=None;wait=0
    for ep in range(max_epochs):
        model.train();perm=torch.randperm(len(Xtr))
        for i in range(0,len(Xtr),batch_size):
            idx=perm[i:i+batch_size]
            xb=Xtr[idx].to(device);yb=ytr[idx].to(device)
            opt.zero_grad()
            g_,n_,a_,b_=model(xb)
            loss=sum(evidential_loss(yb[:,k],g_[:,k],n_[:,k],a_[:,k],b_[:,k]) for k in range(4))
            loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.0);opt.step()
        model.eval();vl=0
        with torch.no_grad():
            for i in range(0,len(Xva),batch_size):
                xb=Xva[i:i+batch_size].to(device);yb=yva[i:i+batch_size].to(device)
                g_,n_,a_,b_=model(xb)
                vl+=sum(evidential_loss(yb[:,k],g_[:,k],n_[:,k],a_[:,k],b_[:,k]) for k in range(4)).item()
        if vl<best-1e-4: best=vl;best_state={kk:v.cpu().clone() for kk,v in model.state_dict().items()};wait=0
        else:
            wait+=1
            if wait>=patience: break
    model.load_state_dict(best_state);model.eval()
    def predict(X):
        gs=[];als=[]
        with torch.no_grad():
            for i in range(0,len(X),batch_size):
                g_,n_,a_,b_=model(X[i:i+batch_size].to(device))
                al=b_/(a_-1)  # aleatoric variance (normalised units)
                gs.append(g_.cpu().numpy());als.append(al.cpu().numpy())
        return np.vstack(gs)*ystd+ymean, np.vstack(als)*(ystd**2)  # de-normalise mean and variance
    pt,pat=predict(Xte);po,pao=predict(Xood)
    return pt,pat,po,pao

print("\ntraining 5 seeds...")
t0=time.time()
te_m=[];te_a=[];ood_m=[];ood_a=[]
for s in range(5):
    pt,pat,po,pao=train_one(seed=SEED+s)
    te_m.append(pt);te_a.append(pat);ood_m.append(po);ood_a.append(pao)
    print("  seed %d done (%.0fs)"%(SEED+s,time.time()-t0))
te_m=np.stack(te_m);te_a=np.stack(te_a);ood_m=np.stack(ood_m);ood_a=np.stack(ood_a)

def ensemble_total_std(means,alea):
    # total predictive variance = between-seed variance of means + mean aleatoric variance
    epi=means.var(axis=0)
    ale=alea.mean(axis=0)
    return means.mean(axis=0), np.sqrt(epi+ale)

def calibrate_and_report(mean,std,idx,cal_mask=None):
    ytrue=y_arr[idx]
    n=len(idx)
    rng2=np.random.RandomState(42)
    perm=rng2.permutation(n);half=n//2
    cal=perm[:half];ev=perm[half:]
    out={}
    for j,t in enumerate(targets):
        yy=ytrue[:,j];pp=mean[:,j];ss=std[:,j]
        def f(sc): return np.mean(np.abs(yy[cal]-pp[cal])<=norm.ppf(0.95)*ss[cal]*sc)-0.90
        try: scale=brentq(f,0.01,50.0)
        except: scale=1.0
        sse=ss[ev]*scale
        p80=float(np.mean(np.abs(yy[ev]-pp[ev])<=norm.ppf(0.9)*sse))
        p90=float(np.mean(np.abs(yy[ev]-pp[ev])<=norm.ppf(0.95)*sse))
        p95=float(np.mean(np.abs(yy[ev]-pp[ev])<=norm.ppf(0.975)*sse))
        zz=np.abs(yy[ev]-pp[ev])/sse;lv=np.linspace(0.05,0.95,10)
        ece=float(np.mean([abs(np.mean(zz<=norm.ppf(0.5+c/2))-c) for c in lv]))
        corr=float(np.corrcoef(np.abs(yy[ev]-pp[ev]),sse)[0,1])
        rmse=float(np.sqrt(np.mean((yy-pp)**2)))
        out[t]=dict(picp80=round(p80,3),picp90=round(p90,3),picp95=round(p95,3),ece10=round(ece,3),err_corr=round(corr,3),rmse=round(rmse,3),scale=round(float(scale),3))
    return out

em_te,es_te=ensemble_total_std(te_m,te_a)
em_ood,es_ood=ensemble_total_std(ood_m,ood_a)
ens_indist=calibrate_and_report(em_te,es_te,te)
ens_ood=calibrate_and_report(em_ood,es_ood,ood_idx)

evidential_indist={"dissolution_av":{"picp80":0.739,"picp90":0.847,"picp95":0.898,"ece10":0.032,"err_corr":-0.069,"rmse":3.344},
"tbl_av_hardness":{"picp80":0.72,"picp90":0.847,"picp95":0.93,"ece10":0.051,"err_corr":0.527,"rmse":7.9},
"tbl_rsd_weight":{"picp80":0.866,"picp90":0.911,"picp95":0.924,"ece10":0.077,"err_corr":0.099,"rmse":0.77},
"fct_tensile":{"picp80":0.847,"picp90":0.93,"picp95":0.968,"ece10":0.118,"err_corr":-0.131,"rmse":0.404}}
evidential_ood={"dissolution_av":{"picp90":1.0,"rmse":2.895},"tbl_av_hardness":{"picp90":0.938,"rmse":18.69},
"tbl_rsd_weight":{"picp90":0.969,"rmse":0.364},"fct_tensile":{"picp90":1.0,"rmse":0.143}}

result={"analysis":"RQ3 deep ensemble vs evidential, fold 1, 5 seeds, calib/eval partition method",
"pkl_md5":md5,"n_seeds":5,"seeds":[SEED+s for s in range(5)],
"config":{"max_epochs":150,"patience":10,"lr":5e-4,"batch_size":16},
"ensemble_uncertainty":"between-seed variance of means plus mean per-seed aleatoric variance, then calibrated on a held-out half",
"in_distribution":{"ensemble":ens_indist,"evidential":evidential_indist},
"ood_code15":{"ensemble":ens_ood,"evidential":evidential_ood}}
with open("/kaggle/working/rq3_ensemble_comparison.json","w") as f: json.dump(result,f,indent=2)

print("\n=== IN-DISTRIBUTION (fold-1 eval half, both calibrated) ===")
print("%-16s %-24s %-24s"%("target","ENSEMBLE p90/ece/corr","EVIDENTIAL p90/ece/corr"))
for t in targets:
    e=ens_indist[t];v=evidential_indist[t]
    print("%-16s %.3f/%.3f/%+.3f        %.3f/%.3f/%+.3f"%(t,e["picp90"],e["ece10"],e["err_corr"],v["picp90"],v["ece10"],v["err_corr"]))
print("\n=== RMSE check (ensemble should be near evidential fold-1) ===")
for t in targets:
    print("  %-16s ensemble %.3f  evidential %.3f"%(t,ens_indist[t]["rmse"],evidential_indist[t]["rmse"]))
print("\nsaved rq3_ensemble_comparison.json  (~%.0f min)"%((time.time()-t0)/60))

device: cuda | Tesla T4
pkl md5: 7d7fc76be1e4940198b76d9d0797a3a9
pkl verified
fold-1 train 610 val 17 test 314 ood 64

training 5 seeds...
  seed 42 done (158s)
  seed 43 done (243s)
  seed 44 done (450s)
  seed 45 done (545s)
  seed 46 done (767s)

=== IN-DISTRIBUTION (fold-1 eval half, both calibrated) ===
target           ENSEMBLE p90/ece/corr    EVIDENTIAL p90/ece/corr 
dissolution_av   0.854/0.042/-0.101        0.847/0.032/-0.069
tbl_av_hardness  0.924/0.071/+0.686        0.847/0.051/+0.527
tbl_rsd_weight   0.911/0.066/+0.125        0.911/0.077/+0.099
fct_tensile      0.904/0.029/+0.189        0.930/0.118/-0.131

=== RMSE check (ensemble should be near evidential fold-1) ===
  dissolution_av   ensemble 3.408  evidential 3.344
  tbl_av_hardness  ensemble 9.081  evidential 7.900
  tbl_rsd_weight   ensemble 0.755  evidential 0.770
  fct_tensile      ensemble 0.404  evidential 0.404

saved rq3_ensemble_comparison.json  (~13 min)
